In [1]:
#Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile

from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import roc_curve, roc_auc_score

In [2]:
#File path to the zipped datafile
zip_path = '../../Data/Processed/Data_Compressed.zip'

#List of filenames to read from the zip file
filenames = [
    'Data_Compressed/all_normalized_features.csv',
    'Data_Compressed/kdd_all_normalized_features.csv',
    'Data_Compressed/kdd_expanded_all_scaled.csv',
    'Data_Compressed/kdd_merged_normalized_all.csv'
]

#Corresponding names for the DataFrames
df_names = [
    'xuetangx_df',
    'kdd_df',
    'kdd_expanded_df',
    'kdd_merged_df'
]

#Create an empty dictionary to store the DataFrames
dataframes = {}

#Open the zip file
z = zipfile.ZipFile(zip_path, 'r')

#Loop through the filenames and read them into DataFrames
for i, file in enumerate(filenames):
    print(f"Reading file: {file} from {zip_path}")
    
    #Read each CSV file directly from the zip
    f = z.open(file)
    dataframes[df_names[i]] = pd.read_csv(f)
    f.close()
    
    print(f"{df_names[i]} loaded with shape: {dataframes[df_names[i]].shape}\n")

#Close the zip file after reading
z.close()

#Assign individual DataFrames to variables
xuetangx_df = dataframes['xuetangx_df']
kdd_df = dataframes['kdd_df']
kdd_expanded_df = dataframes['kdd_expanded_df']
kdd_merged_df = dataframes['kdd_merged_df']


Reading file: Data_Compressed/all_normalized_features.csv from ../../Data/Processed/Data_Compressed.zip
xuetangx_df loaded with shape: (225642, 30)

Reading file: Data_Compressed/kdd_all_normalized_features.csv from ../../Data/Processed/Data_Compressed.zip
kdd_df loaded with shape: (200904, 17)

Reading file: Data_Compressed/kdd_expanded_all_scaled.csv from ../../Data/Processed/Data_Compressed.zip
kdd_expanded_df loaded with shape: (120542, 142)

Reading file: Data_Compressed/kdd_merged_normalized_all.csv from ../../Data/Processed/Data_Compressed.zip
kdd_merged_df loaded with shape: (120542, 158)



In [3]:
# %pip install tensorflow
# %pip install keras
# %pip install scikit-learn
# %pip install fastai
# %pip install torch

In [4]:
#Import required libraries for Deep Learning
#Keras DNN Classifier
from keras.models import Sequential
from keras.layers import BatchNormalization, Dense, Dropout, Input
from keras.regularizers import l2
from keras.utils import to_categorical, normalize
from keras import backend as K

#FastAI DL Classifier
import torch
from fastai.tabular.all import *

#Metrics for evaluation
from sklearn.metrics import balanced_accuracy_score, accuracy_score, precision_score, recall_score, roc_auc_score, f1_score
from sklearn.model_selection import train_test_split

### Data Preprocessing

In [5]:
#Display the info of the xuetangx_df DataFrame
xuetangx_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 225642 entries, 0 to 225641
Data columns (total 30 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   enroll_id                      225642 non-null  int64  
 1   action_count                   225642 non-null  float64
 2   seek_video_count               225642 non-null  float64
 3   play_video_count               225642 non-null  float64
 4   pause_video_count              225642 non-null  float64
 5   stop_video_count               225642 non-null  float64
 6   load_video_count               225642 non-null  float64
 7   problem_get_count              225642 non-null  float64
 8   problem_check_count            225642 non-null  float64
 9   problem_save_count             225642 non-null  float64
 10  reset_problem_count            225642 non-null  float64
 11  problem_check_correct_count    225642 non-null  float64
 12  problem_check_incorrect_count 

In [6]:
#Display the info of the kdd_df DataFrame
kdd_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200904 entries, 0 to 200903
Data columns (total 17 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   enrollment_id             200904 non-null  int64  
 1   action_count              200904 non-null  float64
 2   server_navigate_count     200904 non-null  float64
 3   server_access_count       200904 non-null  float64
 4   server_problem_count      200904 non-null  float64
 5   server_page_close_count   200904 non-null  float64
 6   server_video_count        200904 non-null  float64
 7   server_discussion_count   200904 non-null  float64
 8   server_wiki_count         200904 non-null  float64
 9   browser_navigate_count    200904 non-null  float64
 10  browser_access_count      200904 non-null  float64
 11  browser_problem_count     200904 non-null  float64
 12  browser_page_close_count  200904 non-null  float64
 13  browser_video_count       200904 non-null  f

In [7]:
#Display the info of the kdd_expanded_df DataFrame
kdd_expanded_df.columns

Index(['Unnamed: 0', 'enrollment_id', 'truth', 'avg_chapter_delays',
       'server_discussion_percent', 'act_cnt_weekDay_01',
       'browser_html_percent', 'parallel_enrollments', 'browser_dictation',
       'act_cnt_day_00',
       ...
       'server_course_percent', 'browser_course_info_percent',
       'browser_course', 'browser_vertical_percent', 'sessions_in_week_1',
       'sessions_in_week_0', 'sessions_in_week_3', 'sessions_in_week_2',
       'sessions_in_week_4', 'browser_about'],
      dtype='object', length=142)

In [8]:
#Isolate the X and y features for the xuetangx dataset
xuetangx_X = xuetangx_df.drop(columns=['truth'])
xuetangx_X = xuetangx_X.drop(columns=['enroll_id'])
xuetangx_y = xuetangx_df['truth']
#Isolate the X and y features for the kdd dataset
kdd_X = kdd_df.drop(columns=['truth'])
kdd_X = kdd_X.drop(columns=['enrollment_id'])
kdd_y = kdd_df['truth']
#Isolate the X and y features for the kdd_expanded 
kdd_expanded_X = kdd_expanded_df.drop(columns=['truth'])
kdd_expanded_X = kdd_expanded_X.drop(columns=['enrollment_id'])
kdd_expanded_X = kdd_expanded_X.drop(columns=['Unnamed: 0'])
kdd_expanded_y = kdd_expanded_df['truth']

In [9]:
#Split the xuetangx dataset into training and testing sets
xuetangx_X_train, xuetangx_X_test, xuetangx_y_train, xuetangx_y_test = train_test_split(xuetangx_X, xuetangx_y, test_size=0.2, random_state=100)

#Split the kdd dataset into training and testing sets
kdd_X_train, kdd_X_test, kdd_y_train, kdd_y_test = train_test_split(kdd_X, kdd_y, test_size=0.2, random_state=100)

#Split the kdd_expanded dataset into training and testing sets
kdd_expanded_X_train, kdd_expanded_X_test, kdd_expanded_y_train, kdd_expanded_y_test = train_test_split(kdd_expanded_X, kdd_expanded_y, test_size=0.2, random_state=100)

### Keras-TensorFlow

#### Xuetangx

In [10]:
#Modify the training and test variables to use the xuetangx dataset for readability
X_train, X_test, y_train, y_test = xuetangx_X_train, xuetangx_X_test, xuetangx_y_train, xuetangx_y_test

In [11]:
#Initialize KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

#Prepare results storage
results = {
    'Accuracy': [],
    'Balanced Accuracy': [],
    'Recall': [],
    'Precision': [],
    'AUC': [],
    'F1 Score': []
}

X_np = X_train.values
y_np = y_train.values

for train_index, val_index in kf.split(X_np):
    X_train_fold, X_val_fold = X_np[train_index], X_np[val_index]
    y_train_fold, y_val_fold = y_np[train_index], y_np[val_index]

    #Build and compile model
    model = Sequential([
        Dense(128, kernel_regularizer=l2(0.001), activation='relu', input_shape=(X_train.shape[1],)),
        BatchNormalization(),
        Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        Dense(len(np.unique(y_train)), activation='softmax')
    ])
    
    model.compile(optimizer='adam', loss='categorical_crossentropy')

    #Convert y to one-hot for training
    y_train_cat = to_categorical(y_train_fold)
    
    #Fit the model
    model.fit(X_train_fold, y_train_cat, epochs=100, verbose=0, batch_size=512)

    #Predict and evaluate
    y_val_probs = model.predict(X_val_fold)
    y_pred = np.argmax(y_val_probs, axis=1)

    #Metrics
    acc = accuracy_score(y_val_fold, y_pred) * 100
    bal_acc = balanced_accuracy_score(y_val_fold, y_pred) * 100
    rec = recall_score(y_val_fold, y_pred, average='weighted') * 100
    prec = precision_score(y_val_fold, y_pred, average='weighted') * 100
    auc = roc_auc_score(y_val_fold, y_pred) * 100
    f1 = f1_score(y_val_fold, y_pred) * 100

    #Store metrics
    results['Accuracy'].append(acc)
    results['Balanced Accuracy'].append(bal_acc)
    results['Recall'].append(rec)
    results['Precision'].append(prec)
    results['AUC'].append(auc)
    results['F1 Score'].append(f1)

#Summarize results
summary = {
    metric: f"{np.mean(vals):.2f} ± {np.std(vals):.2f}"
    for metric, vals in results.items()
}



c:\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


KeyboardInterrupt: 

In [ ]:
#Store the results in the comparison DataFrame
xuetangx_tf_df = pd.DataFrame(list(summary.items()), columns=['Metric', 'Value'])
xuetangx_final_df = xuetangx_tf_df.set_index('Metric').T
xuetangx_final_df.index = ['Keras-Tensorflow'] 
print(xuetangx_final_df)

Metric                Accuracy Balanced Accuracy  ...           AUC      F1 Score
Keras-Tensorflow  83.72 ± 0.09      72.35 ± 0.67  ...  72.35 ± 0.67  89.78 ± 0.06

[1 rows x 6 columns]


#### KDD (Experiment 1)

In [ ]:
#Modify the training and test variables to use the xuetangx dataset for readability
X_train, X_test, y_train, y_test = kdd_X_train, kdd_X_test, kdd_y_train, kdd_y_test

In [ ]:
#Initialize KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

#Prepare results storage
results = {
    'Accuracy': [],
    'Balanced Accuracy': [],
    'Recall': [],
    'Precision': [],
    'AUC': [],
    'F1 Score': []
}

X_np = X_train.values
y_np = y_train.values

for train_index, val_index in kf.split(X_np):
    X_train_fold, X_val_fold = X_np[train_index], X_np[val_index]
    y_train_fold, y_val_fold = y_np[train_index], y_np[val_index]

    #Build and compile model
    model = Sequential([
        Dense(128, kernel_regularizer=l2(0.001), activation='relu', input_shape=(X_train.shape[1],)),
        BatchNormalization(),
        Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        Dense(len(np.unique(y_train)), activation='softmax')
    ])
    
    model.compile(optimizer='adam', loss='categorical_crossentropy')

    #Convert y to one-hot for training
    y_train_cat = to_categorical(y_train_fold)
    
    #Fit the model
    model.fit(X_train_fold, y_train_cat, epochs=100, verbose=0, batch_size=512)

    #Predict and evaluate
    y_val_probs = model.predict(X_val_fold)
    y_pred = np.argmax(y_val_probs, axis=1)

    #Metrics
    acc = accuracy_score(y_val_fold, y_pred) * 100
    bal_acc = balanced_accuracy_score(y_val_fold, y_pred) * 100
    rec = recall_score(y_val_fold, y_pred, average='weighted') * 100
    prec = precision_score(y_val_fold, y_pred, average='weighted') * 100
    auc = roc_auc_score(y_val_fold, y_pred) * 100
    f1 = f1_score(y_val_fold, y_pred) * 100

    #Store metrics
    results['Accuracy'].append(acc)
    results['Balanced Accuracy'].append(bal_acc)
    results['Recall'].append(rec)
    results['Precision'].append(prec)
    results['AUC'].append(auc)
    results['F1 Score'].append(f1)

#Summarize results
summary = {
    metric: f"{np.mean(vals):.2f} ± {np.std(vals):.2f}"
    for metric, vals in results.items()
}

c:\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1005/1005 ━━━━━━━━━━━━━━━━━━━━ 1s 521us/step


c:\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1005/1005 ━━━━━━━━━━━━━━━━━━━━ 1s 523us/step


c:\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1005/1005 ━━━━━━━━━━━━━━━━━━━━ 1s 507us/step


c:\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1005/1005 ━━━━━━━━━━━━━━━━━━━━ 1s 606us/step


c:\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1005/1005 ━━━━━━━━━━━━━━━━━━━━ 1s 528us/step


In [ ]:
#Store the results in the comparison DataFrame
kdd_tf_df = pd.DataFrame(list(summary.items()), columns=['Metric', 'Value'])
kdd_final_df = kdd_tf_df.set_index('Metric').T
kdd_final_df.index = ['Keras-Tensorflow'] 
print(kdd_final_df)

Metric                Accuracy Balanced Accuracy  ...           AUC      F1 Score
Keras-Tensorflow  86.02 ± 0.10      72.30 ± 0.47  ...  72.30 ± 0.47  91.56 ± 0.06

[1 rows x 6 columns]


#### KDD (Experiment 2)

In [ ]:
#Modify the training and test variables to use the xuetangx dataset for readability
X_train, X_test, y_train, y_test = kdd_expanded_X_train, kdd_expanded_X_test, kdd_expanded_y_train, kdd_expanded_y_test

In [ ]:
#Initialize KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

#Prepare results storage
results = {
    'Accuracy': [],
    'Balanced Accuracy': [],
    'Recall': [],
    'Precision': [],
    'AUC': [],
    'F1 Score': []
}

X_np = X_train.values
y_np = y_train.values

for train_index, val_index in kf.split(X_np):
    X_train_fold, X_val_fold = X_np[train_index], X_np[val_index]
    y_train_fold, y_val_fold = y_np[train_index], y_np[val_index]

    #Build and compile model
    model = Sequential([
        Dense(128, kernel_regularizer=l2(0.001), activation='relu', input_shape=(X_train.shape[1],)),
        BatchNormalization(),
        Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        Dense(len(np.unique(y_train)), activation='softmax')
    ])
    
    model.compile(optimizer='adam', loss='categorical_crossentropy')

    #Convert y to one-hot for training
    y_train_cat = to_categorical(y_train_fold)
    
    #Fit the model
    model.fit(X_train_fold, y_train_cat, epochs=100, verbose=0, batch_size=512)

    #Predict and evaluate
    y_val_probs = model.predict(X_val_fold)
    y_pred = np.argmax(y_val_probs, axis=1)

    #Metrics
    acc = accuracy_score(y_val_fold, y_pred) * 100
    bal_acc = balanced_accuracy_score(y_val_fold, y_pred) * 100
    rec = recall_score(y_val_fold, y_pred, average='weighted') * 100
    prec = precision_score(y_val_fold, y_pred, average='weighted') * 100
    auc = roc_auc_score(y_val_fold, y_pred) * 100
    f1 = f1_score(y_val_fold, y_pred) * 100

    #Store metrics
    results['Accuracy'].append(acc)
    results['Balanced Accuracy'].append(bal_acc)
    results['Recall'].append(rec)
    results['Precision'].append(prec)
    results['AUC'].append(auc)
    results['F1 Score'].append(f1)

#Summarize results
summary = {
    metric: f"{np.mean(vals):.2f} ± {np.std(vals):.2f}"
    for metric, vals in results.items()
}



c:\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


603/603 ━━━━━━━━━━━━━━━━━━━━ 0s 645us/step


c:\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


603/603 ━━━━━━━━━━━━━━━━━━━━ 0s 582us/step


c:\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


603/603 ━━━━━━━━━━━━━━━━━━━━ 0s 673us/step


c:\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


603/603 ━━━━━━━━━━━━━━━━━━━━ 0s 642us/step


c:\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


603/603 ━━━━━━━━━━━━━━━━━━━━ 0s 586us/step


In [ ]:
#Store the results in the comparison DataFrame
kdd_expanded_tf_df = pd.DataFrame(list(summary.items()), columns=['Metric', 'Value'])
kdd_expanded_final_df = kdd_expanded_tf_df.set_index('Metric').T
kdd_expanded_final_df.index = ['Keras-Tensorflow'] 
print(kdd_expanded_final_df)

Metric                Accuracy Balanced Accuracy  ...           AUC      F1 Score
Keras-Tensorflow  85.66 ± 0.15      73.71 ± 0.53  ...  73.71 ± 0.53  91.24 ± 0.11

[1 rows x 6 columns]


### FastAI

#### Xuetangx

In [ ]:
#Modify the training and test variables to use the xuetangx dataset for readability
X_train, X_test, y_train, y_test = xuetangx_X_train, xuetangx_X_test, xuetangx_y_train, xuetangx_y_test

In [ ]:
#Create a DNN model using FastAI
#Initialize results dictionary first
results = {
    'Accuracy': [],
    'Balanced Accuracy': [],
    'Recall': [],
    'Precision': [],
    'AUC': [],
    'F1 Score': []
}

splits = RandomSplitter(valid_pct=0.2)(range_of(X_train))
X = X_train.copy()
X['truth'] = y_train.values
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, val_idx in kf.split(X):
    train_df = X.iloc[train_idx].copy()
    val_df = X.iloc[val_idx].copy()

    #FastAI Tabular setup
    splits = (list(range(len(train_df))), list(range(len(train_df), len(train_df) + len(val_df))))
    full_df = pd.concat([train_df, val_df])
    
    tp = TabularPandas(
        full_df,
        procs=[],
        cont_names=list(X_train.columns),
        y_names='truth',
        splits=splits
    )

    dls = tp.dataloaders(bs=64)

    learn = tabular_learner(dls, metrics=accuracy)
    learn.fit_one_cycle(5)
    
    # Predictions
    X_val = val_df.drop(columns='truth')
    y_true = val_df['truth'].values
    dl_test = learn.dls.test_dl(X_val)
    
    preds = learn.get_preds(dl=dl_test)[0]
    
    y_pred = (preds >= 0.5).int().numpy()
    
    #Metrics
    results['Accuracy'].append(accuracy_score(y_true, y_pred) * 100)
    results['Balanced Accuracy'].append(balanced_accuracy_score(y_true, y_pred) * 100)
    results['Recall'].append(recall_score(y_true, y_pred, average='weighted') * 100)
    results['Precision'].append(precision_score(y_true, y_pred, average='weighted') * 100)
    results['AUC'].append(roc_auc_score(y_true, y_pred) * 100)
    results['F1 Score'].append(f1_score(y_true, y_pred) * 100)
    results['F1 Score'].append(f1_score(y_true, y_pred) * 100)

#Summarize results
summary = {
    metric: f"{np.mean(vals):.2f} ± {np.std(vals):.2f}"
    for metric, vals in results.items()
}



epoch,train_loss,valid_loss,accuracy,time
0,0.133201,0.145244,0.239758,00:11
1,0.122958,0.180868,0.239758,00:10
2,0.124399,0.463385,0.239758,00:10
3,0.121406,0.208323,0.239758,00:11
4,0.121833,0.250519,0.239758,00:11


epoch,train_loss,valid_loss,accuracy,time
0,0.132328,0.133503,0.241393,00:11
1,0.120886,0.143128,0.241393,00:11
2,0.126940,0.350662,0.241393,00:11
3,0.125170,0.272767,0.241393,00:11
4,0.117660,1.690843,0.241393,00:12


epoch,train_loss,valid_loss,accuracy,time
0,0.130910,0.142035,0.243886,00:11
1,0.126802,0.137838,0.243886,00:12
2,0.124976,0.185364,0.243886,00:12
3,0.121629,0.184617,0.243886,00:11
4,0.122850,0.272847,0.243886,00:11


epoch,train_loss,valid_loss,accuracy,time
0,0.131689,0.141839,0.240873,00:11
1,0.128379,0.135827,0.240873,00:10
2,0.123883,0.133861,0.240873,00:11
3,0.122346,0.128351,0.240873,00:13
4,0.123794,0.127436,0.240873,00:11


epoch,train_loss,valid_loss,accuracy,time
0,0.138816,0.252635,0.244419,00:12
1,0.126727,0.152851,0.244419,00:11
2,0.125832,0.172152,0.244419,00:11
3,0.117975,0.151560,0.244419,00:11
4,0.117751,0.151241,0.244419,00:12


In [ ]:
#Store the results in the comparison DataFrame
xuetangx_fai_df = pd.DataFrame(list(summary.items()), columns=['Metric', 'Value'])
xuetangx_fai_df = xuetangx_fai_df.set_index('Metric').T
xuetangx_fai_df.index = ['Fast.AI']
#Append the results to the existing DataFrame
xuetangx_final_df = pd.concat([xuetangx_final_df, xuetangx_fai_df], axis=0)
print(xuetangx_final_df)

Metric                Accuracy Balanced Accuracy  ...           AUC      F1 Score
Keras-Tensorflow  83.72 ± 0.09      72.35 ± 0.67  ...  72.35 ± 0.67  89.78 ± 0.06
Fast.AI           83.34 ± 0.21      69.85 ± 0.83  ...  69.85 ± 0.83  89.73 ± 0.09

[2 rows x 6 columns]


#### KDD (Experiment 1)

In [ ]:
#Modify the training and test variables to use the kdd dataset for readability
X_train, X_test, y_train, y_test = kdd_X_train, kdd_X_test, kdd_y_train, kdd_y_test

In [ ]:
#Create a DNN model using FastAI
#Initialize results dictionary first
results = {
    'Accuracy': [],
    'Balanced Accuracy': [],
    'Recall': [],
    'Precision': [],
    'AUC': [],
    'F1 Score': []
}

splits = RandomSplitter(valid_pct=0.2)(range_of(X_train))
X = X_train.copy()
X['truth'] = y_train.values
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, val_idx in kf.split(X):
    train_df = X.iloc[train_idx].copy()
    val_df = X.iloc[val_idx].copy()

    #FastAI Tabular setup
    splits = (list(range(len(train_df))), list(range(len(train_df), len(train_df) + len(val_df))))
    full_df = pd.concat([train_df, val_df])
    
    tp = TabularPandas(
        full_df,
        procs=[],
        cont_names=list(X_train.columns),
        y_names='truth',
        splits=splits
    )

    dls = tp.dataloaders(bs=64)

    learn = tabular_learner(dls, metrics=accuracy)
    learn.fit_one_cycle(5)
    
    #Predictions
    X_val = val_df.drop(columns='truth')
    y_true = val_df['truth'].values
    dl_test = learn.dls.test_dl(X_val)
    
    preds = learn.get_preds(dl=dl_test)[0]
    
    #For binary classification
    y_pred = (preds >= 0.5).int().numpy()
    
    #Metrics
    results['Accuracy'].append(accuracy_score(y_true, y_pred) * 100)
    results['Balanced Accuracy'].append(balanced_accuracy_score(y_true, y_pred) * 100)
    results['Recall'].append(recall_score(y_true, y_pred, average='weighted') * 100)
    results['Precision'].append(precision_score(y_true, y_pred, average='weighted') * 100)
    results['AUC'].append(roc_auc_score(y_true, y_pred) * 100)
    results['F1 Score'].append(f1_score(y_true, y_pred) * 100)
    results['F1 Score'].append(f1_score(y_true, y_pred) * 100)

#Summarize results
summary = {
    metric: f"{np.mean(vals):.2f} ± {np.std(vals):.2f}"
    for metric, vals in results.items()
}

epoch,train_loss,valid_loss,accuracy,time
0,0.119227,0.122371,0.208275,00:10
1,0.107561,0.110819,0.208275,00:09
2,0.113021,0.107506,0.208275,00:09
3,0.106209,0.108179,0.208275,00:09
4,0.111423,0.106423,0.208275,00:09


epoch,train_loss,valid_loss,accuracy,time
0,0.111655,0.142875,0.209208,00:09
1,0.107574,0.111573,0.209208,00:09
2,0.107009,0.111107,0.209208,00:09
3,0.105778,0.110102,0.209208,00:09
4,0.108393,0.109011,0.209208,00:09


epoch,train_loss,valid_loss,accuracy,time
0,0.112154,0.142699,0.202022,00:09
1,0.113933,0.109164,0.202022,00:09
2,0.105645,0.108868,0.202022,00:09
3,0.104661,0.107076,0.202022,00:10
4,0.104984,0.105942,0.202022,00:10


epoch,train_loss,valid_loss,accuracy,time
0,0.110972,0.130384,0.210833,00:09
1,0.112827,0.108188,0.210833,00:09
2,0.104615,0.108067,0.210833,00:09
3,0.112486,0.107809,0.210833,00:09
4,0.105688,0.107486,0.210833,00:09


epoch,train_loss,valid_loss,accuracy,time
0,0.122485,0.140914,0.206944,00:10
1,0.106953,0.110062,0.206944,00:10
2,0.111869,0.111301,0.206944,00:11
3,0.106884,0.111207,0.206944,00:10
4,0.108861,0.107954,0.206944,00:10


In [ ]:
#Store the results in the comparison DataFrame
kdd_fai_df = pd.DataFrame(list(summary.items()), columns=['Metric', 'Value'])
kdd_fai_df = kdd_fai_df.set_index('Metric').T
kdd_fai_df.index = ['Fast.AI']
#Append the results to the existing DataFrame
kdd_final_df = pd.concat([kdd_final_df, kdd_fai_df], axis=0)
print(kdd_final_df)

Metric                Accuracy Balanced Accuracy  ...           AUC      F1 Score
Keras-Tensorflow  86.02 ± 0.10      72.30 ± 0.47  ...  72.30 ± 0.47  91.56 ± 0.06
Fast.AI           85.93 ± 0.13      71.87 ± 0.60  ...  71.87 ± 0.60  91.53 ± 0.07

[2 rows x 6 columns]


#### KDD (Experiment 2)

In [ ]:
#Modify the training and test variables to use the kdd_expanded dataset for readability
X_train, X_test, y_train, y_test = kdd_expanded_X_train, kdd_expanded_X_test, kdd_expanded_y_train, kdd_expanded_y_test

In [ ]:
#Create a DNN model using FastAI
#Initialize results dictionary first
results = {
    'Accuracy': [],
    'Balanced Accuracy': [],
    'Recall': [],
    'Precision': [],
    'AUC': [],
    'F1 Score': []
}

splits = RandomSplitter(valid_pct=0.2)(range_of(X_train))
X = X_train.copy()
X['truth'] = y_train.values
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, val_idx in kf.split(X):
    train_df = X.iloc[train_idx].copy()
    val_df = X.iloc[val_idx].copy()

    #FastAI Tabular setup
    splits = (list(range(len(train_df))), list(range(len(train_df), len(train_df) + len(val_df))))
    full_df = pd.concat([train_df, val_df])
    
    tp = TabularPandas(
        full_df,
        procs=[],
        cat_names=[],  # If you have categorical features, add them here
        cont_names=list(X_train.columns),
        y_names='truth',
        splits=splits
    )

    dls = tp.dataloaders(bs=64)

    learn = tabular_learner(dls, metrics=accuracy)
    learn.fit_one_cycle(5)
    
    # Predictions
    X_val = val_df.drop(columns='truth')
    y_true = val_df['truth'].values
    dl_test = learn.dls.test_dl(X_val)
    
    # Fix: get_preds returns a tuple, extract just the predictions
    preds = learn.get_preds(dl=dl_test)[0]
    
    # For binary classification
    y_pred = (preds >= 0.5).int().numpy()
    
    #Metrics
    results['Accuracy'].append(accuracy_score(y_true, y_pred) * 100)
    results['Balanced Accuracy'].append(balanced_accuracy_score(y_true, y_pred) * 100)
    results['Recall'].append(recall_score(y_true, y_pred, average='weighted') * 100)
    results['Precision'].append(precision_score(y_true, y_pred, average='weighted') * 100)
    results['AUC'].append(roc_auc_score(y_true, y_pred) * 100)
    results['F1 Score'].append(f1_score(y_true, y_pred) * 100)
    results['F1 Score'].append(f1_score(y_true, y_pred) * 100)

#Summarize results
summary = {
    metric: f"{np.mean(vals):.2f} ± {np.std(vals):.2f}"
    for metric, vals in results.items()
}

epoch,train_loss,valid_loss,accuracy,time
0,0.100011,0.118652,0.210919,00:06
1,0.101837,0.113130,0.210919,00:06
2,0.100125,0.108777,0.210919,00:06
3,0.095099,0.102234,0.210919,00:06
4,0.098152,0.100613,0.210919,00:07


epoch,train_loss,valid_loss,accuracy,time
0,0.108478,143.275467,0.204853,00:06
1,0.101664,20.676546,0.204853,00:06
2,0.096938,34.951035,0.204853,00:06
3,0.093704,46.164055,0.204853,00:06
4,0.094056,39.196526,0.204853,00:06


epoch,train_loss,valid_loss,accuracy,time
0,0.104977,32.881874,0.206564,00:07
1,0.097489,138.974091,0.206564,00:08
2,0.095530,0.379843,0.206564,00:07
3,0.094933,0.170958,0.206564,00:07
4,0.093356,0.098705,0.206564,00:08


epoch,train_loss,valid_loss,accuracy,time
0,0.111149,231.244354,0.202375,00:08
1,0.111755,180.597794,0.202375,00:07
2,0.100705,40.560574,0.202375,00:07
3,0.096931,56.205383,0.202375,00:07
4,0.094022,65.238800,0.202375,00:07


epoch,train_loss,valid_loss,accuracy,time
0,0.108232,0.114363,0.210204,00:08
1,0.104743,0.107988,0.210204,00:07
2,0.101354,0.102866,0.210204,00:07
3,0.092836,0.095794,0.210204,00:07
4,0.092646,0.099281,0.210204,00:07


In [ ]:
#Store the results in the comparison DataFrame
kdd_expanded_fai_df = pd.DataFrame(list(summary.items()), columns=['Metric', 'Value'])
kdd_expanded_fai_df = kdd_expanded_fai_df.set_index('Metric').T
kdd_expanded_fai_df.index = ['Fast.AI']
#Append the results to the existing DataFrame
kdd_expanded_final_df = pd.concat([kdd_expanded_final_df, kdd_expanded_fai_df], axis=0)
print(kdd_expanded_final_df)

Metric                Accuracy Balanced Accuracy  ...           AUC      F1 Score
Keras-Tensorflow  85.66 ± 0.15      73.71 ± 0.53  ...  73.71 ± 0.53  91.24 ± 0.11
Fast.AI           87.44 ± 0.18      75.77 ± 0.51  ...  75.77 ± 0.51  92.36 ± 0.11

[2 rows x 6 columns]


In [ ]:
#Display the final comparison DataFrame for all three datasets
display(xuetangx_final_df)
display(kdd_final_df)
display(kdd_expanded_final_df)

Metric,Accuracy,Balanced Accuracy,Recall,Precision,AUC,F1 Score
Keras-Tensorflow,83.72 ± 0.09,72.35 ± 0.67,83.72 ± 0.09,82.84 ± 0.09,72.35 ± 0.67,89.78 ± 0.06
Fast.AI,83.34 ± 0.21,69.85 ± 0.83,83.34 ± 0.21,82.69 ± 0.06,69.85 ± 0.83,89.73 ± 0.09


Metric,Accuracy,Balanced Accuracy,Recall,Precision,AUC,F1 Score
Keras-Tensorflow,86.02 ± 0.10,72.30 ± 0.47,86.02 ± 0.10,85.11 ± 0.10,72.30 ± 0.47,91.56 ± 0.06
Fast.AI,85.93 ± 0.13,71.87 ± 0.60,85.93 ± 0.13,85.02 ± 0.13,71.87 ± 0.60,91.53 ± 0.07


Metric,Accuracy,Balanced Accuracy,Recall,Precision,AUC,F1 Score
Keras-Tensorflow,85.66 ± 0.15,73.71 ± 0.53,85.66 ± 0.15,84.76 ± 0.16,73.71 ± 0.53,91.24 ± 0.11
Fast.AI,87.44 ± 0.18,75.77 ± 0.51,87.44 ± 0.18,86.75 ± 0.19,75.77 ± 0.51,92.36 ± 0.11


In [ ]:
folder_path = '../../Analysis/Deep_Learning_Results'
xuetangx_final_df.to_csv(os.path.join(folder_path, 'xuetangx_final_results.csv'))
kdd_final_df.to_csv(os.path.join(folder_path, 'kdd_final_results.csv'))
kdd_expanded_final_df.to_csv(os.path.join(folder_path, 'kdd_expanded_final_results.csv'))

## SVM

Implemented for team members due to having more computational resources. Results are saved as csv files in the `Analysis` folder for members to append to their own results.

### Xuetangx SVM

In [10]:
#Modify the training and test variables to use the xuetangx dataset for readability
X_train, X_test, y_train, y_test = xuetangx_X_train, xuetangx_X_test, xuetangx_y_train, xuetangx_y_test

In [11]:
#Define the model
svc = SVC(gamma='auto')
#Define the KFold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
#Perform cross-validation
scores = cross_val_score(svc, X_train, y_train, cv=kf, scoring='accuracy', n_jobs=-1)
#Calculate the mean and standard deviation of the scores
mean_acc = np.mean(scores)
std_acc = np.std(scores)

In [16]:
#Store the results into a dataframe
xuetangx_svm_df = pd.DataFrame([{
    'Classifiers': 'SVM',
    'Mean Accuracy': round(mean_acc, 5),
    'SD': round(std_acc, 5)
}])

In [17]:
#Save the results to a new CSV file
folder_path = '../../Analysis/SVM_Results'
xuetangx_svm_df.to_csv(os.path.join(folder_path, 'xuetangx_svm_results.csv'))

#### KDD (Experiment 1) SVM

In [ ]:
#Modify the training and test variables to use the kdd dataset for readability
X_train, X_test, y_train, y_test = kdd_X_train, kdd_X_test, kdd_y_train, kdd_y_test

In [ ]:
#Define the model
svc = SVC(gamma='auto')
#Define the KFold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
#Perform cross-validation
scores = cross_val_score(svc, X_train, y_train, cv=kf, scoring='accuracy', n_jobs=-1)
#Calculate the mean and standard deviation of the scores
mean_acc = np.mean(scores)
std_acc = np.std(scores)

In [ ]:
#Store the results into a dataframe
kdd_svm_df = pd.DataFrame([{
    'Classifiers': 'SVM',
    'Mean Accuracy': round(mean_acc, 5),
    'SD': round(std_acc, 5)
}])

In [ ]:
#Save the results to a new CSV file
folder_path = '../../Analysis/SVM_Results'
kdd_svm_df.to_csv(os.path.join(folder_path, 'kdd_svm_results.csv'))

### KDD (Experiment 2) SVM

In [ ]:
#Modify the training and test variables to use the kdd_expanded dataset for readability
X_train, X_test, y_train, y_test = kdd_expanded_X_train, kdd_expanded_X_test, kdd_expanded_y_train, kdd_expanded_y_test

In [ ]:
#Define the model
svc = SVC(gamma='auto')
#Define the KFold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
#Perform cross-validation
scores = cross_val_score(svc, X_train, y_train, cv=kf, scoring='accuracy', n_jobs=-1)
#Calculate the mean and standard deviation of the scores
mean_acc = np.mean(scores)
std_acc = np.std(scores)

In [ ]:
#Store the results into a dataframe
kdd_expanded_svm_df = pd.DataFrame([{
    'Classifiers': 'SVM',
    'Mean Accuracy': round(mean_acc, 5),
    'SD': round(std_acc, 5)
}])

In [ ]:
#Save the results to a new CSV file
folder_path = '../../Analysis/SVM_Results'
kdd_expanded_svm_df.to_csv(os.path.join(folder_path, 'kdd_expanded_svm_results.csv'))